# Brain MRI Detection — Google Colab Setup

Use a **Google Colab Tesla T4 GPU** runtime: **Runtime → Change runtime type → T4 GPU**.
This notebook checks the runtime and prepares storage paths for future detection work.
Codex Cloud is for coding, review, and lightweight CPU checks.

No dataset download, MRI reading, checkpoint loading, model construction, or training is performed.
Keep patient data and weights in private storage; do not commit them or save sensitive cell outputs to GitHub.


## 1. Configure setup

Keep the default GPU requirement in Colab. For a lightweight CPU check, set
`REQUIRE_GPU = False`, `INSTALL_COLAB_DEPENDENCIES = False`, and `MOUNT_GOOGLE_DRIVE = False`.
Edit `PROJECT_ROOT` to your private project folder. Optional overrides can point to an existing dataset
and a future detection checkpoint. Missing data or weights do not prevent setup checks.


In [ ]:
from pathlib import Path

REQUIRE_GPU = True  # @param {type:"boolean"}
INSTALL_COLAB_DEPENDENCIES = True  # @param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
CREATE_OUTPUT_DIRS = False  # @param {type:"boolean"}

DRIVE_MOUNT_POINT = "/content/drive"  # @param {type:"string"}
PROJECT_ROOT = "/content/drive/MyDrive/BrainMRI"  # @param {type:"string"}
DATASET_ROOT_OVERRIDE = ""  # @param {type:"string"}
CHECKPOINT_PATH_OVERRIDE = ""  # @param {type:"string"}

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Google Colab runtime:", IN_COLAB)


## 2. Check PyTorch, CUDA, and GPU

Use Colab's preinstalled CUDA-enabled PyTorch. This cell checks a tiny tensor operation on the selected
device; it does not create a model. If CUDA is unavailable, select a T4 runtime and reconnect.


In [ ]:
import sys
import torch

def check_torch_runtime(torch_module, require_gpu=True):
    cuda_available = torch_module.cuda.is_available()
    if require_gpu and not cuda_available:
        raise RuntimeError(
            "CUDA is unavailable. In Colab select Runtime > Change runtime type > "
            "T4 GPU and reconnect. Use REQUIRE_GPU=False only for CPU setup checks."
        )

    device = torch_module.device("cuda:0" if cuda_available else "cpu")
    with torch_module.no_grad():
        sample = torch_module.arange(4, dtype=torch_module.float32, device=device)
        if (sample * sample).sum().item() != 14.0:
            raise RuntimeError("PyTorch device arithmetic check failed.")
    if cuda_available:
        torch_module.cuda.synchronize()

    info = {
        "python": sys.version.split()[0],
        "pytorch": str(torch_module.__version__),
        "pytorch_cuda_build": torch_module.version.cuda,
        "cuda_available": cuda_available,
        "device": str(device),
        "gpu_name": torch_module.cuda.get_device_name(0) if cuda_available else None,
        "gpu_memory_gib": (
            round(torch_module.cuda.get_device_properties(0).total_memory / 2**30, 2)
            if cuda_available else None
        ),
        "device_tensor_check": "PASS",
    }
    return info

runtime_info = check_torch_runtime(torch, require_gpu=REQUIRE_GPU)
for key, value in runtime_info.items():
    print(f"{key}: {value}")
if runtime_info["gpu_name"] and "T4" not in runtime_info["gpu_name"]:
    print("This runtime has a different GPU; the project default is Tesla T4.")


## 3. Prepare MONAI and NiBabel

Only in Colab, install **MONAI 1.5.1** and **NiBabel 5.3.2**. PyTorch, CUDA, and NumPy remain supplied
by Colab; `--no-deps` prevents this step from replacing them. The next cell checks the installed
package requirements and runs small in-memory compatibility checks. If a requirement fails, stop
and use a compatible Colab runtime. If either library was already imported at a different version,
restart the session and rerun the notebook.


In [ ]:
import importlib.metadata as metadata
import subprocess

COLAB_PACKAGES = {"monai": "1.5.1", "nibabel": "5.3.2"}

if IN_COLAB and INSTALL_COLAB_DEPENDENCIES:
    requested = []
    for package, expected_version in COLAB_PACKAGES.items():
        try:
            current_version = metadata.version(package)
        except metadata.PackageNotFoundError:
            current_version = None
        if current_version != expected_version:
            requested.append(f"{package}=={expected_version}")
    if requested:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check", "--no-input", "--no-deps", *requested,
        ])
    else:
        print("Pinned Colab packages are already installed.")
else:
    print("Dependency installation skipped; existing packages will be checked.")


In [ ]:
import numpy as np
import monai
import nibabel as nib
from packaging.requirements import Requirement
from monai.transforms import NormalizeIntensity

def check_declared_requirements(package_names):
    problems = []
    for package in package_names:
        for requirement_text in metadata.requires(package) or []:
            requirement = Requirement(requirement_text)
            if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
                continue
            try:
                installed_version = metadata.version(requirement.name)
            except metadata.PackageNotFoundError:
                problems.append(f"{package}: missing {requirement.name}")
                continue
            if requirement.specifier and not requirement.specifier.contains(installed_version):
                problems.append(
                    f"{package}: {requirement.name} {installed_version} "
                    f"does not satisfy {requirement.specifier}"
                )
    if problems:
        raise RuntimeError("Package compatibility check failed:\n" + "\n".join(problems))

check_declared_requirements(["torch", "monai", "nibabel"])
for package, module in [("monai", monai), ("nibabel", nib)]:
    if module.__version__ != metadata.version(package):
        raise RuntimeError(f"{package} was updated after import; restart the Colab session.")
    if IN_COLAB and INSTALL_COLAB_DEPENDENCIES:
        if module.__version__ != COLAB_PACKAGES[package]:
            raise RuntimeError(f"Unexpected {package} version; restart and rerun setup.")

# Artificial arrays only; no MRI files or weights are read.
synthetic = np.arange(64, dtype=np.float32).reshape(4, 4, 4)
normalized = NormalizeIntensity()(synthetic)
normalized_tensor = torch.as_tensor(normalized)
assert tuple(normalized_tensor.shape) == (4, 4, 4)
assert torch.isfinite(normalized_tensor).all().item()
assert abs(normalized_tensor.mean().item()) < 1e-5

nifti = nib.Nifti1Image(synthetic, np.eye(4))
assert nifti.shape == (4, 4, 4)
np.testing.assert_array_equal(np.asarray(nifti.dataobj), synthetic)

compatibility_info = {
    "numpy": np.__version__,
    "monai": monai.__version__,
    "nibabel": nib.__version__,
    "declared_requirements": "PASS",
    "monai_transform": "PASS",
    "nifti_in_memory": "PASS",
}
for key, value in compatibility_info.items():
    print(f"{key}: {value}")


## 4. Optionally mount Google Drive

Mounting requests access through Colab's own authorization prompt. Do not put credentials in this
notebook. To work without Drive, disable mounting and change `PROJECT_ROOT` to a local Colab path
such as `/content/brain-mri-detection`. Local Colab files are temporary.


In [ ]:
DRIVE_MOUNTED = False
if MOUNT_GOOGLE_DRIVE:
    if not IN_COLAB:
        print("Drive mounting skipped: this is not a Google Colab runtime.")
    else:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT, force_remount=False)
        DRIVE_MOUNTED = (Path(DRIVE_MOUNT_POINT) / "MyDrive").is_dir()
        if not DRIVE_MOUNTED:
            raise RuntimeError("Google Drive MyDrive is unavailable after mounting.")
else:
    print("Google Drive mounting disabled.")


## 5. Prepare dataset, checkpoint, and output paths

The default dataset path follows the existing MSD Task01 folder convention. The checkpoint path is
a separate **future detection checkpoint**, not a segmentation checkpoint. Edit the overrides in
step 1 to use your own private storage. Only existence is checked; missing paths are reported without
failing setup. No data is downloaded, scanned, or loaded.

Optional directory creation creates only the detection checkpoint and output directories.
It is disabled by default.


In [ ]:
def prepare_paths(project_root, dataset_override="", checkpoint_override="", create_outputs=False):
    root = Path(project_root).expanduser()
    if not root.is_absolute():
        raise ValueError("PROJECT_ROOT must be an absolute path.")
    dataset_root = (
        Path(dataset_override).expanduser() if dataset_override
        else root / "datasets" / "Task01_BrainTumour" / "Task01_BrainTumour"
    )
    checkpoint_path = (
        Path(checkpoint_override).expanduser() if checkpoint_override
        else root / "checkpoints" / "detection" / "detection_baseline.pt"
    )
    output_dir = root / "outputs" / "detection"
    if not dataset_root.is_absolute() or not checkpoint_path.is_absolute():
        raise ValueError("Dataset and checkpoint overrides must be absolute paths.")
    if create_outputs:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        output_dir.mkdir(parents=True, exist_ok=True)
    return {
        "project_root": root,
        "dataset_root": dataset_root,
        "checkpoint_path": checkpoint_path,
        "output_dir": output_dir,
    }

if CREATE_OUTPUT_DIRS and MOUNT_GOOGLE_DRIVE and not DRIVE_MOUNTED:
    raise RuntimeError("Mount Google Drive before creating its output directories.")

paths = prepare_paths(
    PROJECT_ROOT, DATASET_ROOT_OVERRIDE, CHECKPOINT_PATH_OVERRIDE, CREATE_OUTPUT_DIRS
)
dataset_available = paths["dataset_root"].is_dir()
checkpoint_available = paths["checkpoint_path"].is_file()
print("Dataset directory available:", dataset_available)
print("Optional detection checkpoint available:", checkpoint_available)
print("Dataset and checkpoint paths configured; no patient files or weights were opened.")


In [ ]:
setup_status = {
    "runtime_checks": runtime_info["device_tensor_check"],
    "package_compatibility": compatibility_info["declared_requirements"],
    "monai_transform": compatibility_info["monai_transform"],
    "nifti_in_memory": compatibility_info["nifti_in_memory"],
    "drive_mounted": DRIVE_MOUNTED,
    "dataset_directory_available": dataset_available,
    "optional_checkpoint_available": checkpoint_available,
}
for key, value in setup_status.items():
    print(f"{key}: {value}")
print("Setup checks completed. No model or training pipeline has been created.")
